In [7]:
import torch 
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from transformers.image_utils import load_image
from pathlib import Path

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
DEVICE

'cuda'

In [4]:
# Load image 
image = load_image('closing_disclosure.webp')

In [5]:
# Initialize processor and model
processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")

In [8]:
# Quantization
# -------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [19]:
from pathlib import Path

model_dir = Path(
    AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        cache_dir=None
    ).config._name_or_path
)

In [ ]:
print(sum(p.numel() for p in model.parameters())) #parameter


157394496


In [ ]:
# Model
# -------------------------------
model = AutoModelForVision2Seq.from_pretrained(
    "ds4sd/SmolDocling-256M-preview",
    quantization_config=bnb_config,
    device_map="auto"
)

In [23]:
#model.print_trainable_parameters()

In [ ]:

# model = AutoModelForVision2Seq.from_pretrained(
#     "ds4sd/SmolDocling-256M-preview",
#     torch_dtype=torch.bfloat16,
#     _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
# ).to(DEVICE)


In [13]:
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    },
]

In [14]:
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [15]:
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

In [16]:
# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

<doctag><section_header_level_1><loc_29><loc_32><loc_155><loc_44>Closing Disclosure</section_header_level_1>
<section_header_level_1><loc_29><loc_52><loc_100><loc_58>Closing Information</section_header_level_1>
<section_header_level_1><loc_29><loc_133><loc_84><loc_140>Loan Terms</section_header_level_1>
<section_header_level_1><loc_29><loc_149><loc_87><loc_155>Loan Amount</section_header_level_1>
<section_header_level_1><loc_29><loc_162><loc_84><loc_168>Interest Rate</section_header_level_1>
<section_header_level_1><loc_29><loc_174><loc_137><loc_180>Monthly Principal & Interest</section_header_level_1>
<section_header_level_1><loc_29><loc_174><loc_137><loc_181>Monthly Principal & Interest</section_header_level_1>
<section_header_level_1><loc_29><loc_188><loc_143><loc_194>See Projected Payments below for your</section_header_level_1>
<section_header_level_1><loc_29><loc_217><loc_107><loc_224>Prepayment Penalty</section_header_level_1>
<section_header_level_1><loc_29><loc_258><loc_117><l

In [17]:
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")


In [18]:
print(doc.export_to_markdown())

## Closing Disclosure

## Closing Information

## Loan Terms

## Loan Amount

## Interest Rate

## Monthly Principal &amp; Interest

## Monthly Principal &amp; Interest

## See Projected Payments below for your

## Prepayment Penalty

## Projected Payments

## Payment Calculation

## See page 4 for details

## Costs at Closing

## Closing Costs

## Cash to Close

## CLOSING DISCLOSURE

## Loan Amount

## Interest Rate

## Monthly Principal &amp; Interest

## Balloon Payment

## Projected Payments

## See page 4 for details

## Costs at Closing

## Closing Costs

## Cash to Close

## CLOSING DISCLOSURE

## Loan Amount

## Interest Rate

## Monthly Principal &amp; Interest

## Prepayment Penalty

## Projected Payments

## See page 4 for details

## Costs at Closing

## Closing Costs

## Cash to Close

## CLOSING DISCLOSURE

## Loan Terms

## Loan Amount

## Interest Rate

## Monthly Principal &amp; Interest

## Estimated Total Monthly Payment

## Prepayment Penalty

## Payment Calculatio